# Benign commoncrawl data crawler
- creates dataset of URLs, where the canonized URL was present in 2 separate crawl groups separated by 4 years
- --> this means that the URL is not ephemeral and is probably not malicious - otherwise it would have been taken down already

In [ ]:
import json
from tqdm import tqdm
from pathlib import Path
import csv
import gzip
from collections import Counter
import pandas as pd

import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from data.utils.common_crawl.common_crawl import (
    cdx_matches,
    load_shard_paths,
    download_shards,
    url_to_surt_hostname,
    generate_url_matching_key
)
from data.utils.general import (
    get_sld,
    load_blocked_domains_from_json
)
from data.utils.external_sort import external_csv_sort

- shard folder should contain .gz shard files from commoncawl

In [ ]:
CHUNK_LINES = 300_000
HELPERS = "helpers"
FINAL_DATASET_PATH = Path("final_benign_dataset.csv")
PRETRAINING_DOMAINS_PATH = Path("../../pretraining_dataset/domain_counts.json")
BLOCKED_DOMAINS = load_blocked_domains_from_json(path=PRETRAINING_DOMAINS_PATH)

SHARDS_CSV_PATH_2021 = Path(f"{HELPERS}/shards_combined_2021.csv")
SHARDS_FOLDER_PATH_2021 = Path("./2021_shards")
SURT_HOSTNAMES_2021 = f"{HELPERS}/surt_hostnames_2021.csv"

SHARDS_CSV_PATH_2025 = Path(f"{HELPERS}/shards_combined_2025.csv")
SHARDS_FOLDER_PATH_2025 = Path("./2025_shards")
SURT_HOSTNAMES_2025 = f"{HELPERS}/surt_hostnames_2025.csv"

# Download old shards
- uncomment to start downloading specified shards

In [ ]:
CRAWL_ID_2025 = "CC-MAIN-2025-30"
BASE_SHARDS_TO_DOWNLOAD_2025 = [
    "00001",
    "00030",
    "00043",
    "00060",
    "00095",
    "00106",
    "00127",
    "00145",
    "00178",
    "00210",
    "00267",
    "00299",
]

downloaded_newly_2025 = download_shards(
    crawl=CRAWL_ID_2025,
    shards=BASE_SHARDS_TO_DOWNLOAD_2025,
    outdir=SHARDS_FOLDER_PATH_2025,
)

# Create raw initial dataset

In [ ]:
def ensure_target_csv(target_path):
    if not target_path.exists():
        with target_path.open("w", newline="") as f:
            csv.writer(f).writerow(["url_matching_key", "url", "shard"])
        return True
    else:
        print("File already exists")
        return False


def iter_lines_with_bar(fp):
    with gzip.open(fp, "rb") as gz, tqdm(
        total=None,  # no known total
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc=fp.name,
    ) as bar:
        for raw in iter(gz.readline, b""):
            bar.update(len(raw))  # advance by decompressed bytes
            yield raw, bar


def process_shard(fp, shard, writer, blocked_domains):
    file_used = file_disc_cdx = file_disc_block = 0

    for raw, bar in iter_lines_with_bar(fp):
        # merged classification + writing
        try:
            text = raw.decode("utf-8", "ignore")
            j = text.find("{")
            if j < 0:
                raise ValueError
            meta = json.loads(text[j:])
        except Exception:
            file_disc_cdx += 1
        else:
            url = meta.get("url")
            if not url or not cdx_matches(meta): # some length based filtering can be added here
                file_disc_cdx += 1
            else:
                sld = get_sld(url)
                if not sld:
                    file_disc_cdx += 1
                elif sld in blocked_domains:
                    file_disc_block += 1
                else:
                    url_matching_key = generate_url_matching_key(url)
                    writer.writerow([url_matching_key, url, shard])
                    file_used += 1

        total_seen = file_used + file_disc_cdx + file_disc_block
        if total_seen and total_seen % CHUNK_LINES == 0:
            bar.set_postfix(
                {
                    "used%": f"{file_used / total_seen * 100:5.1f}%",
                    "disc(cdx)%": f"{file_disc_cdx / total_seen * 100:5.1f}%",
                    "disc(blkd)%": f"{file_disc_block / total_seen * 100:5.1f}%",
                }
            )

    return file_used, file_disc_cdx, file_disc_block


def create_csv_from_shards(target_path, shard_folder_path, blocked_domains, append_those=[]):
    if not ensure_target_csv(target_path) and len(append_those) == 0:
        return

    files = load_shard_paths(folder_path=shard_folder_path)
    # Filter files only if append_those is not empty
    if append_those:
        files = [(fp, year, month, shard) for fp, year, month, shard in files if shard in append_those]
        print(f"Filtering to {len(files)} shards from append_those list")
    else:
        print(f"Processing all {len(files)} shards")
    files.sort(key=lambda x: x[1])
    print(f"Found {len(files)} shards")

    used = disc_cdx = disc_block = 0

    with target_path.open("a", newline="") as csvfile:
        writer = csv.writer(csvfile)
        for fp, year, month, shard in files:
            file_used, file_disc_cdx, file_disc_block = process_shard(fp, shard, writer, blocked_domains)
            csvfile.flush()
            used += file_used
            disc_cdx += file_disc_cdx
            disc_block += file_disc_block

    print("Done — eligible URLs appended to", target_path)

In [ ]:
if not SHARDS_CSV_PATH_2025.exists():
    create_csv_from_shards(target_path=SHARDS_CSV_PATH_2025, shard_folder_path=SHARDS_FOLDER_PATH_2025, blocked_domains=BLOCKED_DOMAINS)

# Get surt hostnames

In [ ]:
def count_surt_hostnames(input_csv_path, output_csv_path, url_column="url_matching_key", chunksize=1_000_000):
    counts = Counter()
    in_path = Path(input_csv_path)
    out_path = Path(output_csv_path)

    # estimate total rows (optional)
    try:
        with open(in_path, "r", encoding="utf-8", errors="ignore") as fh:
            total_rows = sum(1 for _ in fh) - 1  # minus header
    except Exception:
        total_rows = None

    reader = pd.read_csv(
        in_path,
        usecols=[url_column],
        chunksize=chunksize,
        dtype={url_column: "string"},
        keep_default_na=False,
    )

    for chunk in tqdm(
        reader,
        total=((total_rows // chunksize + 1) if total_rows and total_rows > 0 else None),
        desc="Processing chunks",
    ):
        if url_column not in chunk.columns:
            raise ValueError(f"Column '{url_column}' not found in input CSV")

        # map → drop Nones/empties
        host_surt = (
            chunk[url_column]
            .map(url_to_surt_hostname)
            .dropna()
            .map(lambda x: x.replace(",", "."))
        )
        host_surt = host_surt[host_surt != ""]

        if not host_surt.empty:
            counts.update(host_surt.tolist())

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        f.write("surt_hostname,count\n")
        for hostname, count in counts.items():
            if hostname:  # skip any empty string just in case
                f.write(f"{hostname},{count}\n")

# Cluster idx
- cluster.idx can be donwloaded from commoncrawl group - see documentation

In [ ]:
from itertools import islice

def chunked_iterable(it, size):
    it = iter(it)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

def cluster_idx_to_csv(input_path, output_path, chunksize=1_000_000):
    if output_path.exists():
        output_path.unlink()
    try:
        total_lines = sum(1 for _ in open(input_path, "r", encoding="utf-8"))
    except:
        total_lines = None
    output_path.parent.mkdir(parents=True, exist_ok=True)
    seen = set()
    with open(input_path, "r", encoding="utf-8") as infile, open(output_path, "w", encoding="utf-8") as outfile:
        outfile.write("surt_hostname,shard\n")
        for chunk in tqdm(chunked_iterable(infile, chunksize), total=(total_lines // chunksize + 1) if total_lines else None, desc="Processing chunks"):
            for line in chunk:
                parts = line.rstrip("\n").split()
                if len(parts) < 3:
                    continue
                surt = parts[0]
                shard = parts[2]
                host = surt.split(")", 1)[0]
                host = host.split(":", 1)[0]
                host = host.replace(",", ".")
                if host in seen:
                    continue
                seen.add(host)
                shard_num = shard.split("-", 1)[1].split(".", 1)[0]
                outfile.write(f"{host},{shard_num}\n")

CLUSTER_IDX_SHARDS_CSV_2025 = Path(f"{HELPERS}/cluster_idx_shards_2025.csv")
CLUSTER_IDX_PATH_2025 = Path(f"{HELPERS}/cluster_2025.idx")
if not Path(CLUSTER_IDX_SHARDS_CSV_2025).exists():
    cluster_idx_to_csv(CLUSTER_IDX_PATH_2025, CLUSTER_IDX_SHARDS_CSV_2025)
CLUSTER_IDX_SHARDS_CSV_2021 = Path(f"{HELPERS}/cluster_idx_shards_2021.csv")
CLUSTER_IDX_PATH_2021 = Path(f"{HELPERS}/cluster_2021.idx")
if not Path(CLUSTER_IDX_SHARDS_CSV_2021).exists():
    cluster_idx_to_csv(CLUSTER_IDX_PATH_2021, CLUSTER_IDX_SHARDS_CSV_2021)

In [ ]:
def aggregate_shard_counts(mapping_csv, count_csv, output_csv):
    df_map = pd.read_csv(mapping_csv, dtype={'surt_hostname':'string','shard':'string'})
    df_count = pd.read_csv(count_csv, dtype={'surt_hostname':'string','count':int})
    df = df_map.merge(df_count, on='surt_hostname', how='inner')
    agg = df.groupby('shard', as_index=False)['count'].sum()
    total = agg['count'].sum()
    agg['%'] = (agg['count'] / total * 100).round(2)
    agg = agg.sort_values('count', ascending=False)
    agg.to_csv(output_csv, index=False)

- calculate which shards from 2021 should we download based on surt counts from 2025

In [ ]:
TO_DOWNLOAD_SHARDS_IN_2021 = Path(f"{HELPERS}/to_download_shard_counts_2021.csv")
if not Path(SURT_HOSTNAMES_2025).exists():
    count_surt_hostnames(input_csv_path=SHARDS_CSV_PATH_2025, output_csv_path=SURT_HOSTNAMES_2025)
if not Path(TO_DOWNLOAD_SHARDS_IN_2021).exists():
    # What shards from 2021 should I download?
    aggregate_shard_counts(CLUSTER_IDX_SHARDS_CSV_2021, SURT_HOSTNAMES_2025, TO_DOWNLOAD_SHARDS_IN_2021)

In [ ]:
df = pd.read_csv(TO_DOWNLOAD_SHARDS_IN_2021)
shards_to_download = df.loc[df["count"] > 10000, "shard"].astype(str).tolist()

CRAWL_ID_2021 = "CC-MAIN-2021-04"

download_shards(
    crawl=CRAWL_ID_2021,
    shards=shards_to_download,
    outdir=SHARDS_FOLDER_PATH_2021,
    workers=1,
)

In [ ]:
if not SHARDS_CSV_PATH_2021.exists():
    create_csv_from_shards(target_path=SHARDS_CSV_PATH_2021, shard_folder_path=SHARDS_FOLDER_PATH_2021, blocked_domains=BLOCKED_DOMAINS)

- calculate the following vice versa

In [ ]:
TO_DOWNLOAD_SHARDS_IN_2025 = Path(f"{HELPERS}/to_download_shard_counts_2025.csv")
if not Path(SURT_HOSTNAMES_2021).exists():
    count_surt_hostnames(input_csv_path=SHARDS_CSV_PATH_2021, output_csv_path=SURT_HOSTNAMES_2021)
if not Path(TO_DOWNLOAD_SHARDS_IN_2025).exists():
    # What shards from 2025 should I download?
    aggregate_shard_counts(CLUSTER_IDX_SHARDS_CSV_2025, SURT_HOSTNAMES_2021, TO_DOWNLOAD_SHARDS_IN_2025)

In [ ]:
df = pd.read_csv(TO_DOWNLOAD_SHARDS_IN_2025)
shards_to_download = df.loc[df["count"] > 230000, "shard"].astype(str).tolist()
download_shards(
    crawl=CRAWL_ID_2025,
    shards=shards_to_download,
    outdir=SHARDS_FOLDER_PATH_2025,
    workers=1,
)

In [ ]:
def unique_shards_pandas(csv_path, chunksize=1_000_000):
    shards = set()
    for chunk in pd.read_csv(
        csv_path,
        usecols=["shard"],
        dtype={"shard": "string"},
        chunksize=chunksize,
        keep_default_na=False,
        engine="c",
    ):
        shards.update(chunk["shard"].unique())
    return shards

used_shards = unique_shards_pandas(SHARDS_CSV_PATH_2025)
all_shards = set([i[3] for i in load_shard_paths(SHARDS_FOLDER_PATH_2025)])
downloaded_newly_2025 = all_shards - used_shards
print(downloaded_newly_2025)
create_csv_from_shards(target_path=SHARDS_CSV_PATH_2025, shard_folder_path=SHARDS_FOLDER_PATH_2025, blocked_domains=BLOCKED_DOMAINS, append_those=downloaded_newly_2025)

In [ ]:
SHARDS_CSV_PATH_SORTED_2021 = Path(f"{HELPERS}/shards_combined_2021_sorted.csv")
external_csv_sort(SHARDS_CSV_PATH_2021, SHARDS_CSV_PATH_SORTED_2021, key_cols="url_matching_key", chunk_rows=500_000)

In [ ]:
SHARDS_CSV_PATH_SORTED_2025 = Path(f"{HELPERS}/shards_combined_2025_sorted.csv")
external_csv_sort(SHARDS_CSV_PATH_2025, SHARDS_CSV_PATH_SORTED_2025, key_cols="url_matching_key", chunk_rows=500_000)

In [ ]:
from typing import Union, Set

def csv_inner_join_urls(
    left_csv: Union[str, os.PathLike],
    right_csv: Union[str, os.PathLike], 
    urls_output: Union[str, os.PathLike],
    join_key: str,
    url_col: str,
    *,
    delimiter: str = ",",
    show_progress: bool = True
) -> None:
    """
    Inner join two sorted CSV files in merge-sort style.
    Collects unique URLs from matching rows.
    
    Args:
        left_csv: First sorted CSV file
        right_csv: Second sorted CSV file  
        urls_output: Output file for collected URLs
        join_key: Column name to join on (same in both files)
        url_col: URL column name (same in both files)
        delimiter: CSV delimiter
        show_progress: Show progress bar
    """
    
    # Count rows for progress
    def count_rows(filepath):
        if not show_progress:
            return None
        with open(filepath, 'r') as f:
            return sum(1 for _ in f) - 1  # subtract header
    
    left_total = count_rows(left_csv)
    right_total = count_rows(right_csv)
    
    with open(left_csv, 'r') as left_f, \
         open(right_csv, 'r') as right_f, \
         open(urls_output, 'w', newline='') as url_f:
        
        left_reader = csv.DictReader(left_f, delimiter=delimiter)
        right_reader = csv.DictReader(right_f, delimiter=delimiter)
        
        # Create URL output writer
        url_writer = csv.writer(url_f, delimiter=delimiter)
        url_writer.writerow(['url'])
        
        # Progress tracking
        pbar = tqdm(total=min(left_total or 0, right_total or 0), 
                   desc="Inner join progress") if show_progress else None
        
        collected_urls: Set[str] = set()
        
        try:
            # Get first rows
            left_row = next(left_reader, None)
            right_row = next(right_reader, None)
            
            while left_row and right_row:
                left_key = left_row[join_key]
                right_key = right_row[join_key]
                
                if left_key < right_key:
                    left_row = next(left_reader, None)
                elif left_key > right_key:
                    right_row = next(right_reader, None)
                else:
                    # Keys match - collect all rows with this key from both sides
                    left_matches = [left_row]
                    right_matches = [right_row]
                    
                    # Collect all left rows with same key
                    while True:
                        next_left = next(left_reader, None)
                        if not next_left or next_left[join_key] != left_key:
                            left_row = next_left
                            break
                        left_matches.append(next_left)
                    
                    # Collect all right rows with same key  
                    while True:
                        next_right = next(right_reader, None)
                        if not next_right or next_right[join_key] != right_key:
                            right_row = next_right
                            break
                        right_matches.append(next_right)
                    
                    # Cartesian product of matches
                    for l_row in left_matches:
                        for r_row in right_matches:
                            # Collect URLs
                            left_url = l_row[url_col].strip()
                            right_url = r_row[url_col].strip()
                            
                            if left_url:
                                collected_urls.add(left_url)
                            if right_url and right_url != left_url:
                                collected_urls.add(right_url)
                    
                    if pbar:
                        pbar.update(len(left_matches))
        
        finally:
            if pbar:
                pbar.close()
        
        # Write unique URLs
        for url in sorted(collected_urls):
            url_writer.writerow([url])
        
        if show_progress:
            print(f"✓ Join complete. {len(collected_urls)} unique URLs collected.")


In [ ]:
csv_inner_join_urls(
    left_csv=SHARDS_CSV_PATH_SORTED_2021,
    right_csv=SHARDS_CSV_PATH_SORTED_2025, 
    urls_output="collected_urls.csv",
    join_key="url_matching_key",
    url_col="url"
)